# Milestone 2 — Phân tích cấu trúc và chuẩn hóa

## Mục tiêu và đầu ra

M2 chuyển các version của M1 thành nội dung có cấu trúc. Pipeline **gathering →
hierarchy → chuẩn hóa và bibliography → khử trùng → validation** tạo ba bảng
`elements`, `hierarchy`, `bibtex`. Manifest theo dõi cả version xử lý được, partial,
empty và lỗi để không làm mất mẫu số.

Chương 6 minh họa compilation path và cây phân cấp bằng ví dụ tổng hợp chạy từ parser
thật. Chương 7 giải thích chuẩn hóa/citation/dedup; chương 8 trình bày phép đo trên
cùng input. Ví dụ tổng hợp được tách khỏi các thống kê corpus bên dưới.

## Kết quả xử lý theo version

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
data = json.loads((Path.cwd().parent / 'report_summary.json').read_text(encoding='utf-8'))
scope, m1, m2 = data['scope'], data['m1'], data['m2']
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.spines.top': False, 'axes.spines.right': False})

In [ ]:
labels = {'success':'Success', 'partial':'Partial', 'empty_no_tex':'Empty có bằng chứng',
          'error':'Lỗi parser/root', 'error_missing_source':'Thiếu nguồn'}
status = pd.DataFrame([
    {'Trạng thái': label, 'Số version': m2['status_counts'].get(key, 0)}
    for key, label in labels.items()
])
status['Tỷ lệ trên mọi version (%)'] = (100 * status['Số version'] / scope['versions']).round(2)
display(Markdown('**Bảng M2.1. Trạng thái parser trên toàn bộ version trong snapshot.**'))
display(status)
ax = status.plot.barh(x='Trạng thái', y='Số version', legend=False,
                     color=['#34785a', '#bd8b25', '#768a9a', '#b5534a', '#9b6259'])
for i, row in enumerate(status.to_dict('records')):
    ax.text(row['Số version'] + .6, i, str(row['Số version']), va='center')
ax.set_xlim(0, max(status['Số version']) * 1.16); ax.set_ylabel('')
plt.tight_layout(); plt.show()
display(Markdown('**Hình M2.1. Kết quả parser theo version.** Partial giữ các phần đã đọc được '
                 'nhưng vẫn cần xử lý diagnostics trước khi xác nhận hoàn tất.'))

In [ ]:
complete_papers = sum(row['success_versions'] + row['empty_versions'] == row['expected_versions']
                      for row in m2['coverage'])
success = m2['status_counts'].get('success', 0)
display(Markdown('**Bảng M2.2. Coverage, số dòng và mức giảm sau khử trùng.**'))
rows = [
    ('Version success / mọi version', f"{success}/{scope['versions']}", f"{success/scope['versions']:.2%}"),
    ('Version success / version có TeX', f"{success}/{m2['versions_with_tex']}", f"{success/m2['versions_with_tex']:.2%}"),
    ('Paper có mọi version success hoặc empty', f"{complete_papers}/{scope['papers']}", 'Đánh giá theo từng paper'),
]
rows += [(f'Số dòng {name}', f'{count:,}', 'Sau xử lý') for name, count in m2['rows'].items()]
rows += [(f'Khử trùng {name}', f"{values['before']:,} → {values['after']:,}", f"Giảm {values['reduction_rate']:.2%}")
         for name, values in m2['deduplication'].items()]
display(pd.DataFrame(rows, columns=['Chỉ tiêu', 'Kết quả', 'Diễn giải']))
display(Markdown(f"Có **{m2['unresolved_cites']:,} citation occurrence chưa resolve** và "
                 f"**{m2['bibliography_conflicts']:,} xung đột trường bibliography** được lưu để kiểm tra. "
                 'Tỷ lệ dedup chỉ áp dụng các phần có before-count đo được; không xác nhận '
                 'mọi version partial đã được đọc trọn vẹn.'))

## Đánh giá kết quả

Kiểm tra năm bảng đạt về schema, khóa và quan hệ graph trên snapshot. Trạng thái
`processing_complete` còn false vì partial/error; giảm số phần tử sau dedup không
thay thế điều kiện này. Các trường hợp root mơ hồ và thiếu nguồn được giữ trong
coverage cùng các version xử lý thành công.

Số đo parser hiện có là **Windows CPU**, không phải benchmark cuối trên Colab. Để
cập nhật phép đo Colab, dùng notebook parser với cùng gói input; không cần crawl lại
dữ liệu chỉ để đo M2. Xem [chương 8](08_parser_benchmark.ipynb) và
[phụ lục tái lập](09_reproducibility.ipynb).